# 04 Beyond Basics

Production-ready OpenUSD techniques that power real pipelines.

## Objectives

- Author and interpolate **primvars** for per-geometry data
- Understand **value resolution** across time samples, defaults, and layers
- Add **custom properties** with namespaces
- Non-destructively prune scenes via **active / inactive** prims
- Organize assets with **model kinds** (group, assembly, component, subcomponent)
- Efficiently **traverse** a stage
- Orient yourself in **Hydra** rendering architecture
- Handle **units** and know which metadata USD reconciles automatically

## Usage

Run the cells top-to-bottom. All output assets are written to `_assets/` next to this notebook. Cells are self-contained; re-running any lesson will overwrite its own `_assets/` files.

In [1]:
import sys
print("Python:", sys.version.split()[0])
from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK.")
from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9
pxr import OK.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/04_beyond_basics/_assets


In [2]:
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded.")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers.")
    _HAVE_LOUSD = False

lousd helpers loaded.


## 4.1  Primvars

Primvars (primitive variables) are special attributes that bind user data onto geometric prims so it is available to shaders and interpolates across a surface under subdivision or shading. Typical uses: UVs, vertex colors, deformation data, arbitrary per-point or per-face values.

Key APIs: `UsdGeom.PrimvarsAPI`, `CreatePrimvar`, `SetInterpolation`, interpolation tokens `constant` / `uniform` / `vertex`.

Source: `docs/beyond-basics/primvars.md`

In [3]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

### Example 1: Primvar interpolation (constant, uniform, vertex)

Builds the same two-quad mesh three times and authors `displayColor` with three interpolation modes.

In [4]:
from pxr import Usd, UsdGeom, Gf

# Create stage and default prim
file_path = "_assets/primvars_displaycolor.usda"
stage = create_new_stage(file_path)
world = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world.GetPrim())

# Two-quad mesh topology (6 points, 2 faces)
mesh_vertex_locs = [
    Gf.Vec3f(-1, 0, 0),
    Gf.Vec3f(0, 0, 0),
    Gf.Vec3f(0, 1, 0),
    Gf.Vec3f(-1, 1, 0),
    Gf.Vec3f(1, 0, 0),
    Gf.Vec3f(1, 1, 0),
]
face_vertex_counts = [4, 4]
face_vertex_indices = [0, 1, 2, 3,  1, 4, 5, 2]

per_prim_color = [Gf.Vec3f(0.5, 0.0, 0.5)]
per_face_colors = [Gf.Vec3f(0.0, 0.0, 1.0), Gf.Vec3f(1.0, 0.0, 0.0)]
per_vertex_colors = [
    Gf.Vec3f(0.0, 0.0, 1.0), Gf.Vec3f(0.5, 0.0, 0.5), Gf.Vec3f(0.5, 0.0, 0.5),
    Gf.Vec3f(0.0, 0.0, 1.0), Gf.Vec3f(1.0, 0.0, 0.0), Gf.Vec3f(1.0, 0.0, 0.0)
    ]

# Define interpolation mode and colors
example_meshes = {
    "PerPrim": {
        "interpolation": UsdGeom.Tokens.constant,
        "colors": per_prim_color
    },
    "PerFace": {
        "interpolation": UsdGeom.Tokens.uniform,
        "colors": per_face_colors
    },
    "PerVertex": {
        "interpolation": UsdGeom.Tokens.vertex,
        "colors": per_vertex_colors
    }
}

for i, (example_mesh, color_details) in enumerate(example_meshes.items()):
    mesh_prim = UsdGeom.Mesh.Define(stage, world.GetPath().AppendPath(example_mesh))
    mesh_prim.CreatePointsAttr(mesh_vertex_locs)
    mesh_prim.CreateFaceVertexCountsAttr(face_vertex_counts)
    mesh_prim.CreateFaceVertexIndicesAttr(face_vertex_indices)
    UsdGeom.XformCommonAPI(mesh_prim).SetTranslate(Gf.Vec3d(i * 2.5, 0, 0))

    mesh_disp_color_primvar = mesh_prim.GetDisplayColorPrimvar()
    mesh_disp_color_primvar.SetInterpolation(color_details["interpolation"])
    mesh_disp_color_primvar.Set(color_details["colors"])

stage.Save()

In [5]:
DisplayUSD("_assets/primvars_displaycolor.usda", show_usd_code=True)

### Example 2: Store rest state and deformation as primvars

Writes two vertex primvars on a quad (`rest_state` and `deformation`), computes new points as `rest_state + deformation`, then time-samples `Mesh.points`.

In [6]:
from pxr import Usd, UsdGeom, Sdf, Gf

# set up time sampling parameters
start_tc = 1
end_tc = 90
time_code_per_second = 30

# create stage and default prim
file_path = "_assets/primvars_mesh_deformation.usda"
stage = create_new_stage(file_path)
stage.SetStartTimeCode(start_tc)
stage.SetEndTimeCode(end_tc)
stage.SetTimeCodesPerSecond(time_code_per_second)
world = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world.GetPrim())

# define base mesh
mesh_vertex_locs = [
    Gf.Vec3f(0, 0, 0),
    Gf.Vec3f(1, 0, 0),
    Gf.Vec3f(1, 1, 0),
    Gf.Vec3f(0, 1, 0)]
face_vertex_counts = [4]
face_vertex_indices = [0, 1, 2, 3]

# create mesh prim
plane = UsdGeom.Mesh.Define(stage, world.GetPath().AppendPath("Plane"))
plane.CreatePointsAttr(mesh_vertex_locs)
plane.CreateFaceVertexCountsAttr(face_vertex_counts)
plane.CreateFaceVertexIndicesAttr(face_vertex_indices)
UsdGeom.XformCommonAPI(plane).SetTranslate(Gf.Vec3d(2, 0, 0))
plane.GetDisplayColorPrimvar().Set([Gf.Vec3f(0.1, 0.8, 0.1)])

# set the rest_state for the mesh
plane_privar_api = UsdGeom.PrimvarsAPI(plane)
plane_privar_api.CreatePrimvar(
    "rest_state",
    Sdf.ValueTypeNames.Float3Array,
    UsdGeom.Tokens.vertex).Set(mesh_vertex_locs)

# set deformation for vertex locations as a primvar
deformation = [
    Gf.Vec3f(0.0, 0.0, 0.0),
    Gf.Vec3f(-0.3, 0.4, 0.0),
    Gf.Vec3f(-0.3, 0.4, 0.0),
    Gf.Vec3f(0.0, 0.0, 0.0),
    ]
plane_privar_api = UsdGeom.PrimvarsAPI(plane)
plane_privar_api.CreatePrimvar(
    "deformation",
    Sdf.ValueTypeNames.Float3Array,
    UsdGeom.Tokens.vertex).Set(deformation)

new_points = [
    p + o for p, o in zip(
        mesh_vertex_locs,
        plane_privar_api.GetPrimvar("deformation").Get())]

print("Original vertex locations:", mesh_vertex_locs)
print("\nDeforming mesh with primvar 'deformation':", deformation)
print("\nNew vertex locations:", new_points)

# Time-sample Mesh.points from rest to deformed
plane_points = plane.GetPointsAttr()
plane_points.Set(mesh_vertex_locs, Usd.TimeCode(start_tc))
plane_points.Set(new_points, Usd.TimeCode(end_tc))

stage.Save()

Original vertex locations: [Gf.Vec3f(0.0, 0.0, 0.0), Gf.Vec3f(1.0, 0.0, 0.0), Gf.Vec3f(1.0, 1.0, 0.0), Gf.Vec3f(0.0, 1.0, 0.0)]

Deforming mesh with primvar 'deformation': [Gf.Vec3f(0.0, 0.0, 0.0), Gf.Vec3f(-0.30000001192092896, 0.4000000059604645, 0.0), Gf.Vec3f(-0.30000001192092896, 0.4000000059604645, 0.0), Gf.Vec3f(0.0, 0.0, 0.0)]

New vertex locations: [Gf.Vec3f(0.0, 0.0, 0.0), Gf.Vec3f(0.699999988079071, 0.4000000059604645, 0.0), Gf.Vec3f(0.699999988079071, 1.399999976158142, 0.0), Gf.Vec3f(0.0, 1.0, 0.0)]


In [7]:
DisplayUSD("_assets/primvars_mesh_deformation.usda", show_usd_code=True)

**What just happened:** you attached per-vertex data to a mesh using `PrimvarsAPI`, stored a rest pose and a delta, and time-sampled `points` so the mesh animates between them. Primvars are the mechanism for any rich per-geometry data you want shaders or downstream tools to consume.

## 4.2  Value Resolution

Value resolution is how OpenUSD picks the final value of a property or metadata by walking composed opinions from strongest to weakest. Composition is cached per-prim, value resolution is not. Metadata normally uses strongest-wins (dictionaries merge per key); relationships use list-edit merging; attributes pull from value clips, time samples, and defaults.

Key APIs: `Usd.TimeCode.Default()`, `Usd.TimeCode.EarliestTime()`, `Attribute.Get(time)`, `SetCustomDataByKey`, sublayers.

Source: `docs/beyond-basics/value-resolution.md`

In [8]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

### Example 1: Attribute value resolution and animation

Shows four sources a scale attribute can resolve from: schema fallback, authored default, authored time samples, and interpolated in-between values.

In [9]:
from pxr import Usd, UsdGeom

# Time settings
start_tc = 1
end_tc = 120
cube_anim_start_tc = 60
mid_t = (cube_anim_start_tc + end_tc) // 2
time_code_per_second = 30

# Stage setup
file_path = "_assets/value_resolution_attr.usda"
stage = Usd.Stage.CreateNew(file_path)
stage.SetTimeCodesPerSecond(time_code_per_second)
stage.SetStartTimeCode(start_tc)
stage.SetEndTimeCode(end_tc)

# World, Default Prim, and Ground
world_xform = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world_xform.GetPrim())
UsdGeom.XformCommonAPI(world_xform).SetRotate((-75, 0, 0))

# Create Ground Cube
ground = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendChild("Ground"))
UsdGeom.XformCommonAPI(ground).SetScale((10, 5, 0.1))
UsdGeom.XformCommonAPI(ground).SetTranslate((0, 0, -0.1))

# Static cube with schema-defined default scale (no scale op authored)
static_default_cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendChild("StaticDefaultCube"))
static_default_cube.GetDisplayColorAttr().Set([(0.2, 0.2, 0.8)])
static_default_cube_xform_api = UsdGeom.XformCommonAPI(static_default_cube)
static_default_cube_xform_api.SetTranslate((8, 0, 1))
UsdGeom.Xformable(static_default_cube).AddScaleOp()  # add scale op but do not author a value

# select a non-default cube scale value
cube_set_scale = (1.5, 1.5, 1.5)

# Static cube with an authored default scale (no time samples)
static_cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendChild("StaticCube"))
static_cube.GetDisplayColorAttr().Set([(0.8, 0.2, 0.2)])
static_cube_xform_api = UsdGeom.XformCommonAPI(static_cube)
static_cube_xform_api.SetScale(cube_set_scale)  # set static_cube scale
static_cube_xform_api.SetTranslate((-8, 0, 1.5))

# Animated cube: same default as StaticCube plus time samples
anim_cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendChild("AnimCube"))
anim_cube.GetDisplayColorAttr().Set([(0.2, 0.8, 0.2)])
anim_cube_xform_api = UsdGeom.XformCommonAPI(anim_cube)
anim_cube_xform_api.SetScale(cube_set_scale)  # SAME as static_cube
anim_cube_xform_api.SetTranslate((0, 0, 1.5))

# Author time samples for scale and translate
# anim_cube_xform_api.SetScale(cube_set_scale, Usd.TimeCode(start_tc))
anim_cube_xform_api.SetScale((2.5, 2.5, 2.5), Usd.TimeCode(cube_anim_start_tc))  # first animated sample
anim_cube_xform_api.SetScale((5, 5, 5), Usd.TimeCode(end_tc))  # last sample
anim_cube_xform_api.SetTranslate((0, 0, 2.5), Usd.TimeCode(cube_anim_start_tc))
anim_cube_xform_api.SetTranslate((0, 0, 5.0), Usd.TimeCode(end_tc))

# Read back using resolved scale values
_, _, default_cube_fallback_scale, _, _ = UsdGeom.XformCommonAPI(static_default_cube).GetXformVectors(Usd.TimeCode.Default())
_, _, static_cube_default_scale, _, _ = static_cube_xform_api.GetXformVectors(Usd.TimeCode.Default())
_, _, anim_cube_default_scale, _, _ = anim_cube_xform_api.GetXformVectors(Usd.TimeCode.Default())
_, _, anim_cube_earliest_scale, _, _ = anim_cube_xform_api.GetXformVectors(Usd.TimeCode.EarliestTime())
_, _, anim_cube_tc1_scale, _, _ = anim_cube_xform_api.GetXformVectors(Usd.TimeCode(start_tc))
_, _, scale_mid, _, _ = anim_cube_xform_api.GetXformVectors(Usd.TimeCode(mid_t))

# Illustrate that Get() is the same as Get(Usd.TimeCode.Default())
no_time_code_is_default = static_cube.GetSizeAttr().Get() == static_cube.GetSizeAttr().Get(Usd.TimeCode.Default())

print(f"When querying a value Get() is the same as Get(Usd.TimeCode.Default()): {no_time_code_is_default}\n")

print(f"Scale - StaticDefaultCube (no authored xformOp:scale -> schema fallback):  {default_cube_fallback_scale}")  # returns identity fallback value.
print(f"Scale - StaticCube (authored default at Default time):  {static_cube_default_scale}")  # returns the user authored default value.
print(f"Scale - AnimCube (authored default at Default time):  {anim_cube_default_scale}")  # returns the user authored default value.
print(f"Scale - AnimCube at EarliestTime t={cube_anim_start_tc}:  {anim_cube_earliest_scale}")  # first authored time sample value
print(f"Scale - AnimCube at t={start_tc} (before first sample, clamped):  {anim_cube_tc1_scale}")  # resolved value prior to authored value.
print(f"Scale - AnimCube at mid_t={mid_t} (interpolated):  {scale_mid}")  # interpolated between samples

stage.Save()

When querying a value Get() is the same as Get(Usd.TimeCode.Default()): True

Scale - StaticDefaultCube (no authored xformOp:scale -> schema fallback):  (1, 1, 1)
Scale - StaticCube (authored default at Default time):  (1.5, 1.5, 1.5)
Scale - AnimCube (authored default at Default time):  (1.5, 1.5, 1.5)
Scale - AnimCube at EarliestTime t=60:  (2.5, 2.5, 2.5)
Scale - AnimCube at t=1 (before first sample, clamped):  (2.5, 2.5, 2.5)
Scale - AnimCube at mid_t=90 (interpolated):  (3.75, 3.75, 3.75)


In [10]:
DisplayUSD("_assets/value_resolution_attr.usda", show_usd_code=True)

### Example 2: CustomData and relationship list-edit resolution

Sublayers two stages and shows dictionary metadata resolving per key by strength while relationship targets list-edit merge.

In [11]:
from pxr import Usd, UsdGeom
import os

# --- Layer 1 (weaker)
layer_1_path = "_assets/value_resolution_layer_1.usda"
layer_1_stage = create_new_stage(layer_1_path)

layer_1_xform = UsdGeom.Xform.Define(layer_1_stage, "/World/XformPrim")
layer_1_xform_prim = layer_1_xform.GetPrim()

# "/World/XformPrim" customData
layer_1_xform_prim.SetCustomDataByKey("source",  "layer_1")
layer_1_xform_prim.SetCustomDataByKey("opinion",  "weak")
layer_1_xform_prim.SetCustomDataByKey("unique_layer_value", "layer_1_unique_value")  # only authored in layer_1

# Relationship contribution from base
look_a = UsdGeom.Xform.Define(layer_1_stage, "/World/Looks/LookA")
layer_1_xform_prim.CreateRelationship("look:targets").AddTarget(look_a.GetPath())
layer_1_stage.Save()

# --- Layer 2 (stronger)
layer_2_path = "_assets/value_resolution_layer_2.usda"
layer_2_stage = create_new_stage(layer_2_path)

layer_2_xform = UsdGeom.Xform.Define(layer_2_stage, "/World/XformPrim")
layer_2_xform_prim = layer_2_xform.GetPrim()

# "/World/XformPrim" customData
layer_2_xform_prim.SetCustomDataByKey("source",  "layer_2")
layer_2_xform_prim.SetCustomDataByKey("opinion",  "strong")

# Relationship contribution from override
look_b = UsdGeom.Xform.Define(layer_2_stage, "/World/Looks/LookB")
layer_2_xform_prim.CreateRelationship("look:targets").AddTarget(look_b.GetPath())
layer_2_stage.Save()

# --- Composed stage. First sublayer listed (layer_2) is strongest
composed_path = "_assets/value_resolution_composed.usda"
composed_stage = create_new_stage(composed_path)
composed_stage.GetRootLayer().subLayerPaths = [os.path.basename(layer_2_path), os.path.basename(layer_1_path)]

xform_prim = composed_stage.GetPrimAtPath("/World/XformPrim")
resolved_custom_data = xform_prim.GetCustomData()

# resolved custom data:
print("Resolved CustomData:")
for key, value in resolved_custom_data.items():
    print(f"- '{key}': '{value}'")

# resolved relationship targets:
targets = xform_prim.GetRelationship("look:targets").GetTargets()
print(f"\nResolved relationship targets: {[str(t) for t in targets]}")  # both LookA and LookB

composed_stage.Save()

# Write out the composed stage to a single file for inspection
explicit_composed_path = '_assets/value_resolution_composed_explicit.usda'
txt = composed_stage.ExportToString(addSourceFileComment=False)
with open(explicit_composed_path, "w") as f:
    f.write(txt)

Resolved CustomData:
- 'opinion': 'strong'
- 'source': 'layer_2'
- 'unique_layer_value': 'layer_1_unique_value'
- 'userDocBrief': 'Concrete prim schema for a transform, which implements Xformable.'

Resolved relationship targets: ['/World/Looks/LookB', '/World/Looks/LookA']


In [12]:
DisplayCode("_assets/value_resolution_composed_explicit.usda")

**What just happened:** `source` and `opinion` resolved from the stronger layer while `unique_layer_value` persisted from the weaker one, and the relationship target list contains both `LookA` and `LookB` thanks to list-editing.

## 4.3  Custom Properties

Custom properties are user-defined attributes or relationships added to prims at runtime without writing a schema. They are the easiest way to extend USD for pipeline metadata, IoT data, manufacturing tags, and more. Namespace prefixes (e.g. `acme:sensor:temperature`) claim ownership and group related fields.

Key APIs: `UsdPrim.CreateAttribute(name, typeName, custom=True)`, `SetDocumentation`.

Source: `docs/beyond-basics/custom-properties.md`

In [13]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

In [14]:
import shutil
import os
# cleanup any existing copy
try:
    shutil.rmtree('_assets/cubebox_a02')
except FileNotFoundError:
    pass
src = '../exercise_content/foundations/cubebox_a02'
if os.path.isdir(src):
    shutil.copytree(src, '_assets/cubebox_a02')
    print("Copied cubebox_a02 asset.")
else:
    os.makedirs('_assets/cubebox_a02', exist_ok=True)
    # Fallback: make a trivial referenceable stub so the reference resolves
    from pxr import Usd, UsdGeom
    stub_path = '_assets/cubebox_a02/cubebox_a02.usd'
    stub_stage = Usd.Stage.CreateNew(stub_path) if not os.path.exists(stub_path) else Usd.Stage.Open(stub_path)
    xf = UsdGeom.Xform.Define(stub_stage, '/cubebox_a02')
    stub_stage.SetDefaultPrim(xf.GetPrim())
    stub_stage.GetRootLayer().Save()
    print("Asset not found; created minimal stub for referencing.")

Asset not found; created minimal stub for referencing.


### Example 1: Creating custom attributes

Add weight, category, and hazard attributes to a referenced package asset.

In [15]:
from pxr import Usd, UsdGeom, Sdf

file_path = "_assets/custom_attributes.usda"
stage: Usd.Stage = create_new_stage(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")
geometry_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, world_xform.GetPath().AppendPath("Packages"))

box_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, geometry_xform.GetPath().AppendPath("Box"))
box_prim: Usd.Prim = box_xform.GetPrim()
box_prim.GetReferences().AddReference("./cubebox_a02/cubebox_a02.usd")

# Create additional attributes for the box prim
weight = box_prim.CreateAttribute("acme:weight", Sdf.ValueTypeNames.Float, custom=True)
category = box_prim.CreateAttribute("acme:category", Sdf.ValueTypeNames.String, custom=True)
hazard = box_prim.CreateAttribute("acme:hazardous_material", Sdf.ValueTypeNames.Bool, custom=True)

# Optionally document your custom property
weight.SetDocumentation("The weight of the package in kilograms.")
category.SetDocumentation("The shopping category for the products this package contains.")
hazard.SetDocumentation("Whether this package contains hazard materials.")

# Set values for the attributes
weight.Set(5.5)
category.Set("Cosmetics")
hazard.Set(False)

# Save the stage
stage.Save()

In [16]:
DisplayUSD("_assets/custom_attributes.usda", show_usd_code=True)

### Example 2: Modifying custom attributes

Re-open the stage and update the weight value.

In [17]:
from pxr import Usd

file_path = "_assets/custom_attributes.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)
box_prim = stage.GetPrimAtPath("/World/Packages/Box")

# Get the weight attribute
weight_attr: Usd.Attribute = box_prim.GetAttribute("acme:weight")
# Set the value of the weight attribute
weight_attr.Set(4.25)

# Print the weight of the box
print("Weight of Box:", weight_attr.Get())

stage.Save()

Weight of Box: 4.25


In [18]:
DisplayUSD("_assets/custom_attributes.usda", show_usd_code=True)

### Example 3: Grouping related properties with namespaces

Model a sensor device with an `acme:sensor:` namespace.

In [19]:
from pxr import Usd, UsdGeom, Sdf

file_path = "_assets/sensor_data.usda"
stage: Usd.Stage = create_new_stage(file_path)

# Create a prim to represent a sensor device
sensor_prim = stage.DefinePrim("/EnvironmentSensor", "Xform")

# Group related sensor readings using namespaces: "acme:sensor:"
# "acme" identifies the organization, "sensor" groups the related properties
temperature = sensor_prim.CreateAttribute("acme:sensor:temperature", Sdf.ValueTypeNames.Float, custom=True)
humidity = sensor_prim.CreateAttribute("acme:sensor:humidity", Sdf.ValueTypeNames.Float, custom=True)
pressure = sensor_prim.CreateAttribute("acme:sensor:pressure", Sdf.ValueTypeNames.Float, custom=True)
timestamp = sensor_prim.CreateAttribute("acme:sensor:timestamp", Sdf.ValueTypeNames.String, custom=True)

# Document the custom properties to describe their purpose and units
temperature.SetDocumentation("Temperature reading in degrees Celsius")
humidity.SetDocumentation("Relative humidity as a percentage (0-100)")
pressure.SetDocumentation("Atmospheric pressure in kilopascals (kPa)")
timestamp.SetDocumentation("ISO 8601 formatted timestamp of the sensor reading")

# Set sensor readings
temperature.Set(22.5)
humidity.Set(45.0)
pressure.Set(101.3)
timestamp.Set("2025-01-09T14:30:00Z")

# Print grouped sensor data
print("Sensor Readings:")
print(f"  Temperature: {temperature.Get()}")
print(f"  Humidity: {humidity.Get()}%")
print(f"  Pressure: {pressure.Get()} kPa")
print(f"  Timestamp: {timestamp.Get()}")

stage.Save()

Sensor Readings:
  Temperature: 22.5
  Humidity: 45.0%
  Pressure: 101.30000305175781 kPa
  Timestamp: 2025-01-09T14:30:00Z


In [20]:
DisplayCode("_assets/sensor_data.usda")

**What just happened:** you authored three custom attributes on a referenced asset, updated one of them non-destructively, and then used nested namespaces to encode structured IoT-style data directly into a USD prim.

## 4.4  Active and Inactive Prims

Deactivating a prim is a reversible, non-destructive prune. The prim and all its descendants disappear from the composed stage until reactivated. This is the preferred way to temporarily remove content from a stage.

Key APIs: `UsdPrim.SetActive(bool)`, `UsdPrim.IsActive()`, default predicate `UsdPrimDefaultPredicate`.

Source: `docs/beyond-basics/active-inactive-prims.md`

In [21]:
from lousd.utils.helperfunctions import create_new_stage

In [22]:
from pxr import Usd, UsdGeom, UsdLux, UsdShade

stage: Usd.Stage = Usd.Stage.CreateNew("_assets/active-inactive.usda")

world: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world.GetPrim())

box: UsdGeom.Xform = UsdGeom.Xform.Define(stage, world.GetPath().AppendPath("Box"))
geo_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, box.GetPath().AppendPath("Geometry"))
box_geo: UsdGeom.Cube = UsdGeom.Cube.Define(stage, geo_scope.GetPath().AppendPath("Cube"))

mat_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, box.GetPath().AppendPath("Materials"))
box_mat: UsdShade.Material = UsdShade.Material.Define(stage, mat_scope.GetPath().AppendPath("BoxMat"))

# Define a new Scope primitive at the path "/World/Environment" on the current stage
env: UsdGeom.Scope = UsdGeom.Scope.Define(stage, world.GetPath().AppendPath("Environment"))

# Define a new DistantLight primitive at the path "/World/Environment/SkyLight" on the current stage
distant_light: UsdLux.DistantLight = UsdLux.DistantLight.Define(stage, env.GetPath().AppendPath("SkyLight"))

stage.Save()

### Example 1: Deactivating a prim subtree

In [23]:
from pxr import Usd

# Open the USD stage from the specified file
file_path = "_assets/active-inactive.usda"
stage = Usd.Stage.Open(file_path)

# Iterate through all the prims on the stage
# Print the state of the stage before deactivation
print("Stage contents BEFORE deactivating:")
for prim in stage.Traverse():
    print(prim.GetPath())

# Get the "/World/Box" prim and deactivate it
box = stage.GetPrimAtPath("/World/Box")
# Passing in False to SetActive() will set the prim as Inactive and passing in True will set the prim as active
box.SetActive(False)

print("\n\nStage contents AFTER deactivating:")
for prim in stage.Traverse():
    print(prim.GetPath())

Stage contents BEFORE deactivating:
/World
/World/Box
/World/Box/Geometry
/World/Box/Geometry/Cube
/World/Box/Materials
/World/Box/Materials/BoxMat
/World/Environment
/World/Environment/SkyLight


Stage contents AFTER deactivating:
/World
/World/Environment
/World/Environment/SkyLight


In [24]:
DisplayUSD("_assets/active-inactive.usda", show_usd_code=True)

**What just happened:** deactivating `/World/Box` pruned it (and all descendants) from the default traversal without touching the scene description on disk. A stronger layer can flip `active=true` to bring it back.

## 4.5  Model Kinds

Kinds are prim-level metadata that classify a prim's role in the model hierarchy (`group`, `assembly`, `component`, `subcomponent`). They are orthogonal to schema type, but they let traversals and tools reason about what parts of a scene are reusable assets.

Key APIs: `Usd.ModelAPI.SetKind`, `Kind.Tokens`, `Usd.PrimIsModel`, `Prim.IsComponent`.

Source: `docs/beyond-basics/model-kinds.md`

In [25]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

### Example 1: Traversal with model kinds

`/World` is a group, `/World/Component` is a component, `/World/Markers` has no kind. A model-only traversal edits the component and leaves the markers alone.

In [26]:
from pxr import Usd, UsdGeom, Kind, Gf

# Create stage and model root
file_path = "_assets/model_kinds_component.usda"
stage = create_new_stage(file_path)
world_xform = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world_xform.GetPrim())

# Make /World a group so children can be models
Usd.ModelAPI(world_xform.GetPrim()).SetKind(Kind.Tokens.group)

# Non-model branch: Markers (utility geometry, no kind)
markers = UsdGeom.Scope.Define(stage, world_xform.GetPath().AppendChild("Markers"))

points = {
    "PointA": Gf.Vec3d(-3, 0, -3), "PointB": Gf.Vec3d(-3, 0, 3),
    "PointC": Gf.Vec3d(3, 0, -3), "PointD": Gf.Vec3d(3, 0, 3)
    }
for name, pos in points.items():
    cone = UsdGeom.Cone.Define(stage, markers.GetPath().AppendChild(name))
    cone.CreateAxisAttr(UsdGeom.Tokens.y)
    UsdGeom.XformCommonAPI(cone).SetTranslate(pos)
    cone.CreateDisplayColorPrimvar().Set([Gf.Vec3f(1.0, 0.85, 0.2)])

# Model branch: a Component we want to place as a unit
component = UsdGeom.Xform.Define(stage, world_xform.GetPath().AppendChild("Component"))
Usd.ModelAPI(component.GetPrim()).SetKind(Kind.Tokens.component)
body = UsdGeom.Cube.Define(stage, component.GetPath().AppendChild("Body"))
body.CreateDisplayColorPrimvar().Set([(0.25, 0.55, 0.85)])
UsdGeom.XformCommonAPI(body).SetScale((3.0, 1.0, 3.0))

# Model-only traversal: affect models, ignore markers
for prim in Usd.PrimRange(stage.GetPseudoRoot(), predicate=Usd.PrimIsModel):
    if prim.IsComponent():
        xformable = UsdGeom.Xformable(prim)
        if xformable:
            UsdGeom.XformCommonAPI(xformable).SetTranslate((0.0, 2.0, 0.0))

# Show which prims were considered models
model_paths = [p.GetPath().pathString for p in Usd.PrimRange(stage.GetPseudoRoot(), predicate=Usd.PrimIsModel)]
print("Model prims seen by traversal:", model_paths)

stage.Save()

Model prims seen by traversal: ['/', '/World', '/World/Component']


In [27]:
DisplayUSD("_assets/model_kinds_component.usda", show_usd_code=True)

**What just happened:** only prims inside the model hierarchy (those with a model kind and ancestors with group/assembly kinds) were yielded by the `Usd.PrimIsModel` predicate, so only the component moved.

## 4.6  Stage Traversal

Traversal walks the scenegraph in depth-first order to query or edit prims. `Usd.Stage.Traverse` starts from the pseudo-root and uses the default predicate; `Usd.PrimRange` takes any starting prim and supports custom predicates (combined via bitwise `&`, `|`, `~`) and subtree pruning via `PruneChildren`.

Key APIs: `Usd.Stage.Traverse`, `Usd.Stage.TraverseAll`, `Usd.PrimRange`, `Usd.PrimIsActive`, `Usd.PrimIsLoaded`, `Usd.PrimIsModel`, `Prim.GetAllChildren`.

Source: `docs/beyond-basics/stage-traversal.md`

In [28]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage


from pxr import Usd, UsdGeom, UsdLux, UsdShade

file_path = "_assets/stage_traversal.usda"
stage: Usd.Stage = create_new_stage(file_path)

world: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world.GetPrim())

box: UsdGeom.Xform = UsdGeom.Xform.Define(stage, world.GetPath().AppendPath("Box"))
geo_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, box.GetPath().AppendPath("Geometry"))
box_geo: UsdGeom.Cube = UsdGeom.Cube.Define(stage, geo_scope.GetPath().AppendPath("Cube"))

mat_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, box.GetPath().AppendPath("Materials"))
box_mat: UsdShade.Material = UsdShade.Material.Define(stage, mat_scope.GetPath().AppendPath("BoxMat"))

# Define a new Scope primitive at the path "/World/Environment" on the current stage
env: UsdGeom.Scope = UsdGeom.Scope.Define(stage, world.GetPath().AppendPath("Environment"))

# Define a new DistantLight primitive at the path "/World/Environment/SkyLight" on the current stage
distant_light: UsdLux.DistantLight = UsdLux.DistantLight.Define(stage, env.GetPath().AppendPath("SkyLight"))

stage.Save()

In [29]:
DisplayCode("_assets/stage_traversal.usda")

### Example 1: `Stage.Traverse()`

In [30]:
# Import the Usd module from the pxr package
from pxr import Usd

# Open the USD stage from the specified file
stage: Usd.Stage = Usd.Stage.Open("_assets/stage_traversal.usda")

# Traverse and print the paths for the visited prims
for prim in stage.Traverse():
    # Print the path of each prim
    print(prim.GetPath())

/World
/World/Box
/World/Box/Geometry
/World/Box/Geometry/Cube
/World/Box/Materials
/World/Box/Materials/BoxMat
/World/Environment
/World/Environment/SkyLight


### Example 2: Filter traversal by prim type

In [31]:
# Import necessary modules from the pxr package
from pxr import Usd, UsdGeom

# Open the USD stage from the specified file
stage: Usd.Stage = Usd.Stage.Open("_assets/stage_traversal.usda")

scope_count = 0
xform_count = 0
# Traverse through each prim in the stage
for prim in stage.Traverse():
    # Check if the prim is of type Scope
    if UsdGeom.Scope(prim):
        scope_count += 1
        print("Scope Type: ", prim.GetName())
    # Check if the prim is of type Xform
    elif UsdGeom.Xform(prim):
        xform_count +=1
        print("Xform Type: ", prim.GetName())

print("Number of Scope prims: ", scope_count)
print("Number of Xform prims: ", xform_count)

Xform Type:  World
Xform Type:  Box
Scope Type:  Geometry
Scope Type:  Materials
Scope Type:  Environment
Number of Scope prims:  3
Number of Xform prims:  2


### Example 3: Iterate children of a specific prim

In [32]:
# Import the `Usd` module from the `pxr` package:
from pxr import Usd

# Open the USD stage from the specified file:
stage: Usd.Stage = Usd.Stage.Open("_assets/stage_traversal.usda")

# Get the default prim of the stage (/World in this case):
default_prim: Usd.Prim = stage.GetDefaultPrim()

# Iterate through all children of the default prim
for child in default_prim.GetAllChildren():
    # Print the path of each child prim
    print(child.GetPath())

/World/Box
/World/Environment


### Example 4: `Usd.PrimRange` starting from a subtree

In [33]:
# Import the Usd module from the pxr package
from pxr import Usd

# Open the USD stage from the specified file
stage: Usd.Stage = Usd.Stage.Open("_assets/stage_traversal.usda")

prim_range = Usd.PrimRange(stage.GetPrimAtPath("/World/Box"))
for prim in prim_range:
    print(prim.GetPath())

/World/Box
/World/Box/Geometry
/World/Box/Geometry/Cube
/World/Box/Materials
/World/Box/Materials/BoxMat


**What just happened:** you iterated the whole stage depth-first, filtered by schema type, walked a prim's direct children, and scoped a traversal to a single subtree. Combine those tools with bitwise predicates for efficient production-scale walks.

## 4.7  Hydra

Hydra is OpenUSD's rendering architecture. It decouples scene description from rendering backends so the same USD stage can be drawn by HdStorm (real-time OpenGL/Metal/Vulkan), Arnold, Renderman, Embree, and more.

Three main parts:

- **Scene delegate** - provides the scene information
- **Render index** - tracks changes and manages the scene
- **Render delegate** - turns the index into backend-specific rendering commands

Source: `docs/beyond-basics/hydra.md`

There are no executable source cells in the Hydra lesson. The cell below is a defensive probe that checks whether `UsdImagingGL` (the GL imaging entry point into Hydra / HdStorm) is available in your environment. Rendering a viewport requires a GL context, so we only import and report.

In [34]:
# Defensive probe: UsdImagingGL is not always installed (e.g. headless / minimal pxr builds).
try:
    from pxr import UsdImagingGL
    print("UsdImagingGL import OK.")
    renderer_plugins = []
    try:
        engine = UsdImagingGL.Engine()
        renderer_plugins = engine.GetRendererPlugins()
    except Exception as e:
        print("Could not construct UsdImagingGL.Engine (likely no GL context):", e)
    if renderer_plugins:
        print("Available Hydra render delegates:")
        for p in renderer_plugins:
            print(" -", p)
    else:
        print("No Hydra render delegate plugins enumerated in this environment.")
except ImportError:
    print("UsdImagingGL is not available in this environment. This is expected on headless installs.")
    print("Hydra concepts still apply: scene delegate + render index + render delegate.")

UsdImagingGL is not available in this environment. This is expected on headless installs.
Hydra concepts still apply: scene delegate + render index + render delegate.


**What just happened:** a purely conceptual lesson, so instead of rendering we probed for Hydra's GL front end and listed any render delegates registered in your build. The import is wrapped in `try/except` so this cell never errors, even on headless or slim installs.

## 4.8  Units in OpenUSD

Stage metadata records the intent of content creators: `metersPerUnit` (linear scale), `upAxis`, `kilogramsPerUnit` (mass), and `timeCodesPerSecond` (temporal scale). During composition USD **automatically** reconciles `timeCodesPerSecond` but **does not** reconcile `metersPerUnit`, `kilogramsPerUnit`, or `upAxis` - those are the assembler's responsibility.

Key APIs: `UsdGeom.SetStageMetersPerUnit`, `UsdGeom.GetStageMetersPerUnit`, `UsdGeom.SetStageUpAxis`, `stage.SetTimeCodesPerSecond`, `UsdPhysics.SetStageKilogramsPerUnit`.

Source: `docs/beyond-basics/units.md`

In [35]:
from lousd.utils.visualization import DisplayUSD, DisplayCode
from lousd.utils.helperfunctions import create_new_stage

### Example 1: metersPerUnit composition behavior

Two 1-meter cubes authored at different scales are referenced into a millimeter-scale scene. USD copies values literally, so the centimeter cube appears 10x too small.

In [36]:
from pxr import Usd, UsdGeom, Sdf, Gf

# Create a cube asset in CENTIMETERS (metersPerUnit = 0.01)
cm_asset_path = "_assets/1m_cube_centimeters.usda"
cm_stage = create_new_stage(cm_asset_path)
UsdGeom.SetStageUpAxis(cm_stage, UsdGeom.Tokens.y)
UsdGeom.SetStageMetersPerUnit(cm_stage, 0.01)  # Centimeters

# Create a 1 meter cube in centimeter coordinate system
cube_in_cm = UsdGeom.Cube.Define(cm_stage, "/Cube")
cube_in_cm.GetSizeAttr().Set(100.0)
cube_in_cm.GetDisplayColorAttr().Set([(0.2, 0.6, 0.9)])

# Position and save
cm_stage.SetDefaultPrim(cube_in_cm.GetPrim())
cm_stage.Save()

print(f"Centimeter asset: size = { cube_in_cm.GetSizeAttr().Get()} at metersPerUnit = {UsdGeom.GetStageMetersPerUnit(cm_stage)}")
print(f"  -> Represents a {cube_in_cm.GetSizeAttr().Get() * UsdGeom.GetStageMetersPerUnit(cm_stage):.1f} meter cube\n")

# Create a cube asset in MILLIMETERS (metersPerUnit = 0.001)
mm_asset_path = "_assets/cube_in_millimeters.usda"
mm_stage = create_new_stage(mm_asset_path)
UsdGeom.SetStageUpAxis(mm_stage, UsdGeom.Tokens.y)
UsdGeom.SetStageMetersPerUnit(mm_stage, 0.001)  # Millimeters

# Create a 1 meter cube in millimeter coordinate system
cube_in_mm = UsdGeom.Cube.Define(mm_stage, "/Cube")
cube_in_mm.GetSizeAttr().Set(1000.0)
cube_in_mm.GetDisplayColorAttr().Set([(0.9, 0.5, 0.2)])

# Position and save
mm_stage.SetDefaultPrim(cube_in_mm.GetPrim())
mm_stage.Save()

print(f"Millimeter asset: size = { cube_in_mm.GetSizeAttr().Get()} at metersPerUnit = {UsdGeom.GetStageMetersPerUnit(mm_stage)}")
print(f"  -> Represents a {cube_in_mm.GetSizeAttr().Get() * UsdGeom.GetStageMetersPerUnit(mm_stage):.1f} meter cube\n")

# Create a scene that references both cubes (using millimeter scale)
scene_path = "_assets/units_mismatch_scene.usda"
scene_stage = create_new_stage(scene_path)
UsdGeom.SetStageUpAxis(scene_stage, UsdGeom.Tokens.y)
UsdGeom.SetStageMetersPerUnit(scene_stage, 0.001)  # Scene is in millimeters

# Add both cube references
world = UsdGeom.Xform.Define(scene_stage, "/World")
scene_stage.SetDefaultPrim(world.GetPrim())

meter_ref = scene_stage.DefinePrim("/World/Cube_1m_In_Centimeters")
meter_ref.GetReferences().AddReference(cm_asset_path)
UsdGeom.XformCommonAPI(meter_ref).SetTranslate(Gf.Vec3d(-1000, 0, 0))

cm_ref = scene_stage.DefinePrim("/World/Cube_1m_In_Millimeters")
cm_ref.GetReferences().AddReference(mm_asset_path)
UsdGeom.XformCommonAPI(cm_ref).SetTranslate(Gf.Vec3d(1000, 0, 0))

scene_stage.Save()

print("Scene composition (metersPerUnit = 0.001):")
print("  - Referenced 1m cube in centimeter-scale appears 10x too small")
print("  - Referenced 1m cube in millimeter-scale appears correctly")

Centimeter asset: size = 100.0 at metersPerUnit = 0.01
  -> Represents a 1.0 meter cube

Millimeter asset: size = 1000.0 at metersPerUnit = 0.001
  -> Represents a 1.0 meter cube

Scene composition (metersPerUnit = 0.001):
  - Referenced 1m cube in centimeter-scale appears 10x too small
  - Referenced 1m cube in millimeter-scale appears correctly


In [37]:
DisplayUSD("_assets/units_mismatch_scene.usda", show_usd_code=True)

### Example 2: Automatic timeCodesPerSecond scaling

Reference a 60 fps animated asset into a 24 fps scene. USD scales the time samples automatically so the 2-second motion survives the composition.

In [38]:
from pxr import Usd, UsdGeom, Gf

# Create animated asset at 60 fps
anim_asset_path = "_assets/animated_60fps.usda"
anim_stage = create_new_stage(anim_asset_path)
UsdGeom.SetStageUpAxis(anim_stage, UsdGeom.Tokens.y)
UsdGeom.SetStageMetersPerUnit(anim_stage, 1.0)
anim_stage.SetTimeCodesPerSecond(60)  # 60 fps
anim_stage.SetStartTimeCode(0)
anim_stage.SetEndTimeCode(120)  # 2 seconds of animation at 60fps

# Create animated sphere
anim_sphere = UsdGeom.Sphere.Define(anim_stage, "/AnimatedSphere")
anim_sphere.GetRadiusAttr().Set(1.0)
anim_sphere.GetDisplayColorAttr().Set([(0.3, 0.9, 0.3)])

# Animate from left to right over 2 seconds (120 frames at 60fps)
xform_api = UsdGeom.XformCommonAPI(anim_sphere)
xform_api.SetTranslate(Gf.Vec3d(-5, 0, 0), Usd.TimeCode(0))
xform_api.SetTranslate(Gf.Vec3d(5, 0, 0), Usd.TimeCode(120))

anim_stage.SetDefaultPrim(anim_sphere.GetPrim())
anim_stage.Save()

print(f"Animation asset: {anim_stage.GetStartTimeCode()}-{anim_stage.GetEndTimeCode()} @ {anim_stage.GetTimeCodesPerSecond()} fps")
print(f"  -> Duration: {(anim_stage.GetEndTimeCode() - anim_stage.GetStartTimeCode() + 1) / anim_stage.GetTimeCodesPerSecond():.1f} seconds")
print(f"  -> Animates from x=-5 to x=5\n")

# Create scene at 24 fps that references the 60fps animation
scene_24fps_path = "_assets/units_timecode_scene.usda"
scene_stage = create_new_stage(scene_24fps_path)
UsdGeom.SetStageUpAxis(scene_stage, UsdGeom.Tokens.y)
UsdGeom.SetStageMetersPerUnit(scene_stage, 1.0)
scene_stage.SetTimeCodesPerSecond(24)  # Scene is 24 fps
scene_stage.SetStartTimeCode(0)
scene_stage.SetEndTimeCode(48)  # 2 seconds at 24fps

world = UsdGeom.Xform.Define(scene_stage, "/World")
scene_stage.SetDefaultPrim(world.GetPrim())

# Create a floor
floor = UsdGeom.Cube.Define(scene_stage, "/World/Plane")
floor_xform_api = UsdGeom.XformCommonAPI(floor)
floor_xform_api.SetTranslate(Gf.Vec3d(0, -1, 0))
floor_xform_api.SetScale(Gf.Vec3f(10.0, 0.1, 4.0))

# Author the same animation in 24fps for comparison
local_anim = UsdGeom.Sphere.Define(scene_stage, "/World/LocalAnimation")
local_anim.GetRadiusAttr().Set(1.0)
local_anim.GetDisplayColorAttr().Set([(0.9, 0.3, 0.3)])

# Animate from left to right over 2 seconds (48 frames at 24fps)
xform_api = UsdGeom.XformCommonAPI(local_anim)
xform_api.SetTranslate(Gf.Vec3d(-5, 2, 0), Usd.TimeCode(0))
xform_api.SetTranslate(Gf.Vec3d(5, 2, 0), Usd.TimeCode(48))

# Reference thte 60fps animated sphere
ref_prim = scene_stage.DefinePrim("/World/ReferencedAnimation")
ref_prim.GetReferences().AddReference(anim_asset_path)

scene_stage.Save()

print(f"Scene: {scene_stage.GetStartTimeCode()}-{scene_stage.GetEndTimeCode()} @ {scene_stage.GetTimeCodesPerSecond()} fps")
print(f"  -> Duration: {(scene_stage.GetEndTimeCode() - scene_stage.GetStartTimeCode() + 1) / scene_stage.GetTimeCodesPerSecond():.1f} seconds")
print(f"  -> Local sphere animates from x=-5 to x=5 like the referenced sphere")
print(f"  -> Referenced animation automatically scaled from 60fps to 24fps")
print(f"  -> Same 2-second duration and motion maintained")

Animation asset: 0.0-120.0 @ 60.0 fps
  -> Duration: 2.0 seconds
  -> Animates from x=-5 to x=5

Scene: 0.0-48.0 @ 24.0 fps
  -> Duration: 2.0 seconds
  -> Local sphere animates from x=-5 to x=5 like the referenced sphere
  -> Referenced animation automatically scaled from 60fps to 24fps
  -> Same 2-second duration and motion maintained


In [39]:
DisplayUSD("_assets/units_timecode_scene.usda", show_usd_code=True)

### Defensive demo: round-tripping stage unit metadata

Short, self-contained demo that sets and reads back every unit field described in the lesson, including the `UsdPhysics.SetStageKilogramsPerUnit` path wrapped so it does not error if `UsdPhysics` is missing.

In [40]:
from pxr import Usd, UsdGeom

demo_path = "_assets/units_metadata_demo.usda"
stage = create_new_stage(demo_path)

# Linear units: switch from default to centimeters
UsdGeom.SetStageMetersPerUnit(stage, 0.01)
print("metersPerUnit:", UsdGeom.GetStageMetersPerUnit(stage))

# Up axis
UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.z)
print("upAxis:", UsdGeom.GetStageUpAxis(stage))

# Time
stage.SetTimeCodesPerSecond(48)
print("timeCodesPerSecond:", stage.GetTimeCodesPerSecond())

# Mass (optional dependency)
try:
    from pxr import UsdPhysics
    UsdPhysics.SetStageKilogramsPerUnit(stage, 0.001)  # grams
    print("kilogramsPerUnit:", UsdPhysics.GetStageKilogramsPerUnit(stage))
except ImportError:
    print("UsdPhysics not available; skipping kilogramsPerUnit demo.")

stage.Save()

metersPerUnit: 0.01
upAxis: Z
timeCodesPerSecond: 48.0


kilogramsPerUnit: 0.001


**What just happened:** you saw USD copy linear values literally across a unit mismatch (dangerous), saw time samples scale automatically (safe), and exercised every unit metadata setter in a tiny round-trip demo.

## Key Takeaways

- **Primvars** bind per-geometry data with well-defined interpolation modes.
- **Value resolution** walks strongest-to-weakest opinions: strongest-wins for most metadata, per-key merging for dictionaries, list-edit merging for relationships, and time-aware lookup for attributes.
- **Custom properties** are the fastest way to extend USD; namespace them to avoid collisions and document intent.
- **Active/inactive** gives you reversible, non-destructive pruning to manage scene complexity.
- **Model kinds** create a queryable model hierarchy so traversals and tools can reason about assets.
- **Stage traversal** with `Usd.PrimRange` + predicates is how you scale to millions of prims.
- **Hydra** cleanly separates scene description from rendering backends.
- **Units**: `timeCodesPerSecond` reconciles automatically during composition, but `metersPerUnit`, `kilogramsPerUnit`, and `upAxis` do not - pipelines must enforce conventions or apply corrective transforms.

## Next up -> 05_creating_composition_arcs.ipynb